In [ ]:
%matplotlib widget

In [ ]:
from glob import glob
import numpy as np
import pandas as pd
import flammkuchen as fl
from split_dataset import SplitDataset
from bouter import Experiment
from fimpy.pipeline.general import calc_f0, dff
from motions.utilities import stim_vel_dir_dataframe, quantize_directions
from scipy.interpolate import interp1d 
from scipy.signal import convolve2d
import colorspacious
import napari
import matplotlib.pyplot as plt

from fimpylab.core.twop_experiment import TwoPExperiment

from pathlib import Path
import tifffile as tiff

from general_analysis.helper_functions_imaging.general_imaging import normalize_traces

In [ ]:
import json

In [ ]:
# make sensory regressors. requires old bouter stimulus_param_log.
def make_sensory_regressors(exp, n_dirs=8, upsampling=5, sampling=1/3):
    stim = stim_vel_dir_dataframe(exp)
    bin_centres, dir_bins = quantize_directions(stim.theta)
    ind_regs = np.zeros((n_dirs, len(stim)))
    for i_dir in range(n_dirs):
        ind_regs[i_dir, :] = (np.abs(dir_bins - i_dir) < 0.1) & (stim.vel > 0.1)  

    dt_upsampled = sampling / upsampling
    t_imaging_up = np.arange(0, stim.t.values[-1], dt_upsampled)
    reg_up = interp1d(stim.t.values, ind_regs, axis=1, fill_value="extrapolate")(
        t_imaging_up
    )
    
    # 6s kernel
    u_steps = t_imaging_up.shape[0]
    u_time = np.arange(u_steps) * dt_upsampled
    decay = np.exp(-u_time / (1.5 / np.log(2)))
    kernel = decay / np.sum(decay)
    
    convolved = convolve2d(reg_up, kernel[None, :])[:, 0:u_steps]
    reg_sensory = convolved[:, ::upsampling]

    return pd.DataFrame(reg_sensory.T, columns=[f"motion_{i}" for i in range(n_dirs)])

In [ ]:
# calculate directional tuning from dF/F traces, px-wise
def get_tuning_map(traces, sens_regs, n_dirs=8):

    n_t = sens_regs.shape[0]
    reg = sens_regs.T @ traces[:n_t, :]
    print(np.shape(reg))
    #reg = reg.reshape(reg.shape[0], traces.shape[-1], traces.shape[-1])
    
    # tuning vector
    bin_centers, bins = quantize_directions([0], n_dirs)
    vectors = np.stack([np.cos(bin_centers), np.sin(bin_centers)], 0)
    print(np.shape(vectors))
    reg_vectors = vectors @ reg
    print(np.shape(reg_vectors))

    angle = np.arctan2(reg_vectors[1], reg_vectors[0])
    amp = np.sqrt(np.sum(reg_vectors ** 2, 0))

    return amp, angle

In [ ]:
# make a color map

def JCh_to_RGB255(x):
    output = np.clip(colorspacious.cspace_convert(x, "JCh", "sRGB1"), 0, 1)
    return (output * 255).astype(np.uint8)

def color_stack(
        amp,
        angle,
        hueshift=2.5,
        amp_percentile=80,
        maxsat=50,
        lightness_min=100,
        lightness_delta=-40,
    ):
    output_lch = np.zeros((amp.shape[0], 3))
    print(np.shape(output_lch))
    output_lch[:,0]
    maxamp = np.percentile(amp, amp_percentile)

    output_lch[:, 0] = (
            lightness_min + (np.clip(amp / maxamp, 0, 1)) * lightness_delta
    )
    output_lch[:, 1] = (np.clip(amp / maxamp, 0, 1)) * maxsat
    output_lch[:, 2] = (angle + hueshift) * 180 / np.pi

    return JCh_to_RGB255(output_lch)

In [ ]:
master = Path(r"Z:\Hagar\s11\e0075")
master = Path(r"Z:\Hagar and Ot\e0075\habenula")

fish_list = list(master.glob("*_f*"))
fish = fish_list[9]
print(fish)

In [ ]:
anatomy = tiff.imread(fish / 'anatomy_from_suite2p.tif')
#anatomy = fl.load(fish / 'data_from_suite2p_unfiltered.h5')['anatomy_stack']


fish_path = fish / 'suite2p'
paths = list(fish_path.glob("*00*"))
print(paths)

In [ ]:
img_exp = TwoPExperiment(fish)
fs = img_exp['imaging']['microscope_config']['scanning']['framerate']
sampling = 1/fs
res = img_exp.resolution
z_res, x_res, y_res = img_exp.resolution

thresh = 0.3
print(z_res)

In [ ]:
img_exp.n_planes

In [ ]:
fig2, axs2 = plt.subplots(1, 1, figsize=(3, 3))
axs2.spines['right'].set_visible(False)
axs2.spines['top'].set_visible(False)
axs2.imshow(np.rot90(np.nanmean(anatomy, axis=0), 3), cmap='gray_r')
#axs2.invert_yaxis()

In [ ]:
n_row = 3
n_col = 3
fig3, axs3 = plt.subplots(n_row, n_col, figsize=(10, 6))

In [ ]:
ind_count = 0
for path in paths[:]:
    exp = glob(str(path / "*behavior*"))[0]
    
    try:
        traces = fl.load(path / "filtered_traces.h5", "/undetr")
    except:
        traces = fl.load(path / "data_from_suite2p_unfiltered.h5", "/traces")
        traces = normalize_traces(traces.T)
    
    coords = fl.load(path / "data_from_suite2p_unfiltered.h5", "/coords")
    in_brain_idx = np.arange(np.shape(coords)[0])#suite2p_brain['coords_idx']
    
    time = np.linspace(0, traces.shape[0]*sampling, traces.shape[0])
    len_rec, num_cells = np.shape(traces)
    
    # make a list of sensory regressors 
    reg = fl.load(path / 'sensory_regressors.h5')['regressors_conv']
    reg_list = [reg]
    n_t = reg.shape[0]
    
    # calculate tuning
    amp, angle = get_tuning_map(traces, reg.T)
    df = pd.DataFrame(list(zip(amp, angle)), columns=["amp", "angle"])
    
    colors = color_stack(amp, angle)
    
    coords_ib = coords#[in_brain_idx]
    colors_ib = colors#[in_brain_idx]
    amp_ib = amp[in_brain_idx]
    amp_norm = amp_ib / np.nanmax(amp_ib)

    amp_thresh = np.copy(amp_ib)
    amp_thresh[np.where(amp_ib < thresh)[0]] *= 0

    colors_thresh = np.copy(colors_ib)
    colors_thresh[np.where(amp_ib < thresh)[0]] *= 0
    colors_thresh[np.where(amp_ib < thresh)[0]] += 220
    
    
    rel_rois = fl.load(path / "reliability_index_arr.h5", '/reliability_arr_combined')
    
    
    selected_vis = np.where(rel_rois > thresh)[0]
    coords_vis = coords_ib[selected_vis]
    colors_vis = colors_ib[selected_vis]
    colors_thresh = colors_thresh[selected_vis]
    amp_vis = amp_ib[selected_vis]
    
    
    # plot tuning maps
    r = ind_count // n_col
    c = np.mod(ind_count, n_col)
    
    mp_ind = np.argsort(amp_vis)
    axs3[r,c].scatter(coords_ib[:,1]*y_res, coords_ib[:,2]*x_res, c='linen', s=5, alpha=0.8)
    axs3[r,c].scatter(coords_vis[mp_ind,1]*y_res, coords_vis[mp_ind,2]*x_res, c=colors_thresh[mp_ind]/255, s=5, alpha=0.8)

    axs3[r,c].spines['right'].set_visible(False)
    axs3[r,c].spines['top'].set_visible(False)
    axs3[r,c].invert_yaxis()
    axs3[r,c].set_title('z' + str(ind_count))

    ind_count += 1

    #axs2.scatter(coords_vis[mp_ind,1]*y_res, coords_vis[mp_ind,2]*x_res, c=colors_thresh[mp_ind]/255, s=5, alpha=0.8)
    axs2.scatter(coords_vis[mp_ind,1], coords_vis[mp_ind,2], c=colors_thresh[mp_ind]/255, s=10, alpha=0.8)

    


In [ ]:
#file_name = "tuning_thresh" + str(thresh) + ".jpg"
#fig3.savefig(fish / file_name, dpi=300)

file_name = "tuning_thresh stack " + str(thresh) + ".jpg"
fig2.savefig(fish / file_name, dpi=300)
file_name = "tuning_thresh stack " + str(thresh) + ".pdf"
fig2.savefig(fish / file_name, dpi=300)

In [ ]:
np.max(amp_ib)

In [ ]:
##### only reliable traces 

In [ ]:
reliable_arr = fl.load(path / "reliable_rois.h5", "/reliability_arr")
rel_thresh = 0.4
selected_vis = np.where(reliable_arr > rel_thresh)[0]

In [ ]:
coords_vis = coords_ib[selected_vis]
colors_vis = colors_ib[selected_vis]
amp_vis = amp_ib[selected_vis]

In [ ]:
fig2, axs2 = plt.subplots(2, 2, figsize=(4, 4), gridspec_kw={'width_ratios': [8, 2], 'height_ratios': [1, 4]})
mp_ind = np.argsort(amp_vis)
axs2[1,0].scatter(coords_ib[:,2]*0.6, coords_ib[:,1]*.6, c='linen', s=2, alpha=0.8)
axs2[1,0].scatter(coords_vis[mp_ind,2]*0.6, coords_vis[mp_ind,1]*.6, c=colors_vis[mp_ind]/255, s=2, alpha=0.8)

axs2[1,1].scatter(coords_ib[:,0]*10, coords_ib[:,1]*0.6, c='linen', s=2, alpha=0.8)
axs2[1,1].scatter(coords_vis[mp_ind,0]*10, coords_vis[mp_ind,1]*0.6, s=2, c=colors_vis[mp_ind]/255, alpha=0.8)

axs2[0,0].scatter(coords_ib[:,2]*0.6, coords_ib[:,0]*10, c='linen', s=2, alpha=0.8)
axs2[0,0].scatter(coords_vis[mp_ind,2]*0.6, coords_vis[mp_ind,0]*10, s=2, c=colors_vis[mp_ind]/255, alpha=0.8)

axs2[0,0].spines['right'].set_visible(False)
axs2[0,0].spines['top'].set_visible(False)
axs2[0,0].invert_xaxis()

axs2[1,1].spines['right'].set_visible(False)
axs2[1,1].spines['top'].set_visible(False)

axs2[1,0].spines['right'].set_visible(False)
axs2[1,0].spines['top'].set_visible(False)
axs2[1,0].invert_xaxis()

axs2[0,0].axis('off')
axs2[0,1].axis('off')
axs2[1,1].axis('off')
axs2[0,1].axis('off')
axs2[1,0].axis('off')
axs2[1,0].set_xlim(500, 0)
axs2[0,0].set_xlim(500, 0)

axs2[1,0].set_ylim(0, 550)
axs2[1,1].set_ylim(0, 550)

In [ ]:
plt.subplots_adjust(wspace=0.1, hspace=0.1)

In [ ]:
file_name = "tuning_reliable_b" + str(rel_thresh) + " 2.pdf"
#fig2.savefig(path / file_name, dpi=300)
file_name = "tuning_reliable_b" + str(rel_thresh) + " 2.jpg"
fig2.savefig(path / file_name, dpi=300)
